In [1]:
import pandas as pd
import numpy as np
df_debt = pd.read_excel("20230521160827_42424.xlsx", skiprows=2)#skip first 2 header rows
#Rename columns (so it's easier to work with)  
df_debt.columns = [
    'Region', 'Province', 'Purpose', 
    'Debt_2547', 'Debt_2549', 'Debt_2550', 'Debt_2552', 'Debt_2554', 
    'Debt_2556', 'Debt_2558', 'Debt_2560', 'Debt_2562', 'Debt_2564', 'Debt_2566'
]
#Forward fill merged cell DON'T DROP
df_debt[['Province', 'Region']] = df_debt[['Province', 'Region']].ffill()
#Drop 2last rows
df_debt = df_debt.drop(df_debt.index[-2:])

#Replace - with 0
df_debt = df_debt.replace('-', 0)
#combine bangkok and vicinity provinces {ใช้ .isin() ในการเปลี่ยนชื่อ Region ของทุกจังหวัดในลิสต์ให้เป็นกลุ่มเดียวกัน}
vicinity_provinces = ['กรุงเทพมหานคร', 'นนทบุรี', 'ปทุมธานี', 'สมุทรปราการ', 'นครปฐม', 'สมุทรสาคร']
df_debt.loc[df_debt['Province'].isin(vicinity_provinces), 'Region'] = 'กรุงเทพมหานครและปริมณฑล'

df_debt_summary = df_debt[df_debt['Purpose'] == 'หนี้สินทั้งสิ้น'].copy()
#Wrangling income
df_income = pd.read_excel("20230521160113_19514.xlsx", skiprows=2) #skip first 2 header rows
df_income = df_income.drop(columns=[df_income.columns[0]]) #drop first column
df_income.columns = [
    'Region', 'Province',
    'Income_2547', 'Income_2549', 'Income_2550', 'Income_2552', 'Income_2554',
    'Income_2556', 'Income_2558', 'Income_2560', 'Income_2562', 'Income_2564', 'Income_2566'
]
df_income[['Region']] = df_income[['Region']].ffill()
#Vicinity provinces for income data
bkk_vicinity = ['กรุงเทพมหานคร', 'นนทบุรี', 'ปทุมธานี', 'สมุทรปราการ', 'นครปฐม', 'สมุทรสาคร']
df_income.loc[df_income['Province'].isin(bkk_vicinity), 'Region'] = 'กรุงเทพมหานครและปริมณฑล'
#drop last row
df_income = df_income.drop(df_income.index[-1:])
#Fill missing values with 0
df_income = df_income.fillna(0)
#clean index
df_income_sorted = df_income.sort_values(by=['Region', 'Province']).reset_index(drop=True)

#remove subtotal rows
df_income_clean = df_income_sorted[df_income_sorted['Region'] != df_income_sorted['Province']].copy()

/var/folders/sn/f6wtqqcs48scby6bfdrb4gn40000gn/T/ipykernel_44734/3255920952.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_debt = df_debt.replace('-', 0)


In [2]:
# ============================================================
# DAY 1 : Label Definition + Feature Engineering
# ใช้ต่อจาก cell ที่คุณมีอยู่แล้ว: df_debt_summary, df_income_clean
# ============================================================


# ------------------------------------------------------------
# STEP 0 : เตรียมข้อมูลตั้งต้น (รวม Debt summary + Income)
# ------------------------------------------------------------
# df_debt_summary  -> มีแค่แถว "หนี้สินทั้งสิ้น" ต่อจังหวัด (ตามที่คุณเลือกแล้ว)
# df_income_clean  -> รายได้เฉลี่ยต่อจังหวัด

df_debt_summary = df_debt_summary.drop(columns=['Purpose'])
df_merged = pd.merge(
    df_income_clean,
    df_debt_summary,
    on=['Province', 'Region']
)

# wide -> long แบบเดียวกับที่คุณทำใน notebook เดิม (pd.wide_to_long)
df_panel = pd.wide_to_long(
    df_merged,
    stubnames=['Income', 'Debt'],
    i=['Region', 'Province'],
    j='Year',
    sep='_'
).reset_index()

# Year ในข้อมูลเป็น พ.ศ. แบบ string/ปนกัน -> แปลงเป็น int ให้แน่ใจ
df_panel['Year'] = df_panel['Year'].astype(int)
df_panel = df_panel.sort_values(['Province', 'Year']).reset_index(drop=True)


# ------------------------------------------------------------
# STEP 1 : คำนวณ DTI (ของปีนั้นๆ) -> ใช้สร้าง LABEL เท่านั้น
# ------------------------------------------------------------
# Annual_Income = รายได้เฉลี่ยต่อเดือน x 12
df_panel['Annual_Income'] = df_panel['Income'] * 12
df_panel['DTI'] = df_panel['Debt'] / df_panel['Annual_Income']

# --- ตั้ง Threshold ---
# DTI > 1.0 หมายถึง หนี้สินเฉลี่ยเกินรายได้ทั้งปี (ใช้ตรงกับ benchmark
# ที่อ้างถึงในงานวิจัยที่ใช้ฐานข้อมูล NSO เดียวกัน และตรงกับกราฟ
# Top-10 ที่คุณทำไว้แล้วในโน้ตบุ๊กเดิม)
DTI_THRESHOLD = 1.0

df_panel['Label'] = (df_panel['DTI'] > DTI_THRESHOLD).astype(int)
# Label = 1 -> High-Risk Distress
# Label = 0 -> Stable

print("สัดส่วน Label ทั้งหมด:")
print(df_panel['Label'].value_counts(normalize=True))
# ดูตรงนี้ก่อน! ถ้า Label=1 มีสัดส่วนน้อยมาก (เช่น <10%) ยืนยันว่า
# เป็น imbalanced classification ตามที่ระบุไว้ใน proposal
# (ต้องใช้ Precision/Recall วัดผล ไม่ใช่ Accuracy เฉยๆ)


# ------------------------------------------------------------
# STEP 2 : สร้าง FEATURE แบบ Leakage-Safe
# ------------------------------------------------------------
# กฎเหล็ก: ห้ามใช้ Debt, Income, DTI ของ "ปีปัจจุบัน" เป็น feature
# เพราะ Label คำนวณมาจากค่าพวกนี้ตรงๆ (= Label leakage)
# สิ่งที่ "ใช้ได้" คือค่าของ "ปีก่อนหน้า" (lag) เพราะเป็นข้อมูลที่
# มีอยู่จริงก่อนจะรู้ผลของปีปัจจุบัน

def add_gap_and_lag(group):
    """ทำงานทีละจังหวัด (group by Province) เรียงตามปีแล้ว shift หาค่า lag"""
    group = group.sort_values('Year').copy()

    group['Year_prev'] = group['Year'].shift(1)
    group['gap'] = group['Year'] - group['Year_prev']  # ช่วงห่างปีไม่เท่ากัน!

    group['Debt_lag1'] = group['Debt'].shift(1)
    group['Income_lag1'] = group['Income'].shift(1)
    group['DTI_lag1'] = group['DTI'].shift(1)

    return group

def annualized_growth(curr, prev, gap):
    with np.errstate(invalid='ignore', divide='ignore', over='ignore'):
        g = (curr / prev) ** (1 / gap) - 1
    return g

def add_gap_lag_and_growth(group):
    group = group.sort_values('Year').copy()

    group['Year_prev'] = group['Year'].shift(1)
    group['gap'] = group['Year'] - group['Year_prev']

    group['Debt_lag1'] = group['Debt'].shift(1)
    group['Income_lag1'] = group['Income'].shift(1)
    group['DTI_lag1'] = group['DTI'].shift(1)

    growth_gap = group['gap'].shift(1)

    group['Debt_growth_ann'] = annualized_growth(
        group['Debt_lag1'],
        group['Debt_lag1'].shift(1),
        growth_gap
    )

    group['Income_growth_ann'] = annualized_growth(
        group['Income_lag1'],
        group['Income_lag1'].shift(1),
        growth_gap
    )

    return group

df_panel = df_panel.groupby('Province', group_keys=False).apply(add_gap_lag_and_growth)
# --- Region encoding ---
df_panel = pd.get_dummies(df_panel, columns=['Region'], prefix='Region')


# ------------------------------------------------------------
# STEP 3 : ตัด Feature ที่ "ห้ามใช้" ออก + ตัดแถวที่ไม่มี lag (ปีแรกของแต่ละจังหวัด)
# ------------------------------------------------------------
LEAKY_COLS = ['Debt', 'Income', 'Annual_Income', 'DTI']  # ของปีปัจจุบัน ห้ามใช้เป็น X

feature_cols = [c for c in df_panel.columns
                 if c not in LEAKY_COLS + ['Label', 'Province', 'Year',
                                            'Year_prev', 'gap']]

df_model = (
    df_panel
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=feature_cols)
    .copy()
)

print("\nFeature columns ที่จะใช้เทรนโมเดล:")
print(feature_cols)


# ------------------------------------------------------------
# STEP 4 : Train/Test Split แบบ Out-of-Time
# ------------------------------------------------------------
TEST_YEAR = 2566  # ปีล่าสุด -> เทส, ปีก่อนหน้าทั้งหมด -> เทรน

train_df = df_model[df_model['Year'] < TEST_YEAR]
test_df = df_model[df_model['Year'] == TEST_YEAR]

X_train, y_train = train_df[feature_cols], train_df['Label']
X_test, y_test = test_df[feature_cols], test_df['Label']

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Train Label distribution:\n", y_train.value_counts(normalize=True))
print("Test Label distribution:\n", y_test.value_counts(normalize=True))

สัดส่วน Label ทั้งหมด:
Label
0    0.968123
1    0.031877
Name: proportion, dtype: float64

Feature columns ที่จะใช้เทรนโมเดล:
['Debt_lag1', 'Income_lag1', 'DTI_lag1', 'Debt_growth_ann', 'Income_growth_ann', 'Region_กรุงเทพมหานครและปริมณฑล', 'Region_ภาคกลาง', 'Region_ภาคตะวันออกเฉียงเหนือ', 'Region_ภาคเหนือ', 'Region_ภาคใต้']

Train shape: (611, 10), Test shape: (77, 10)
Train Label distribution:
 Label
0    0.963993
1    0.036007
Name: proportion, dtype: float64
Test Label distribution:
 Label
0    0.974026
1    0.025974
Name: proportion, dtype: float64


/var/folders/sn/f6wtqqcs48scby6bfdrb4gn40000gn/T/ipykernel_44734/2172817397.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_panel = df_panel.groupby('Province', group_keys=False).apply(add_gap_lag_and_growth)


In [3]:
df_merged.head(30)

,Region,Province,Income_2547,Income_2549,Income_2550,Income_2552,Income_2554,Income_2556,Income_2558,Income_2560,...,Debt_2549,Debt_2550,Debt_2552,Debt_2554,Debt_2556,Debt_2558,Debt_2560,Debt_2562,Debt_2564,Debt_2566
0,กรุงเทพมหานครและปริมณฑล,กรุงเทพมหานคร,29842.68,36658.0,39020.0,42379.830480,48951.0,49190.8,45571.7,45707.31,...,158059.0,155396.0,207665.0,218741.4,275576.8,164705.6,202699.85,180297.49,191011.02,161049.80
1,กรุงเทพมหานครและปริมณฑล,นครปฐม,20701.35,28980.0,25447.0,24988.667244,22954.8,30855.5,40347.0,32760.69,...,167798.0,100110.0,101030.0,59484.4,175893.9,153504.0,75904.34,105576.01,139849.50,168027.21
2,กรุงเทพมหานครและปริมณฑล,นนทบุรี,26657.97,31152.0,32743.0,34626.276126,35119.7,30663.6,36884.0,40860.88,...,179007.0,196895.0,240769.0,258853.4,260752.2,277605.7,288940.49,248010.00,330809.71,267542.71
3,กรุงเทพมหานครและปริมณฑล,ปทุมธานี,21529.67,25143.0,26107.0,26686.223015,21615.5,33461.3,41056.9,41483.71,...,193153.0,152187.0,220761.0,145293.0,386957.4,221748.2,294901.20,288109.83,370530.90,362493.05
4,กรุงเทพมหานครและปริมณฑล,สมุทรปราการ,19946.54,20382.0,21302.0,23359.377965,23797.9,29575.3,25457.2,28711.77,...,95975.0,85904.0,117361.0,115168.8,207741.1,87647.2,179803.61,123125.61,179708.18,211961.77
5,กรุงเทพมหานครและปริมณฑล,สมุทรสาคร,15346.96,19555.0,18735.0,20978.086026,20850.2,23658.3,29347.4,25446.01,...,138320.0,69569.0,133793.0,104821.6,56367.4,128830.4,104677.28,139201.11,221942.04,82662.76
6,ภาคกลาง,กาญจนบุรี,12121.83,15606.0,15326.0,17571.299299,15209.9,18746.0,18883.8,20564.74,...,114158.0,91113.0,107527.0,68785.6,105780.2,89861.9,157164.45,178100.52,232180.83,183407.88
7,ภาคกลาง,จันทบุรี,15896.75,20606.0,18866.0,19441.859738,24277.6,27283.5,36023.5,32893.60,...,176403.0,186301.0,125646.0,124265.5,110504.3,130258.2,186071.96,144098.32,174221.25,248279.60
8,ภาคกลาง,ฉะเชิงเทรา,16937.66,16770.0,20665.0,21251.572120,23030.7,34548.3,27554.9,26061.85,...,77611.0,151115.0,95398.0,102745.4,217297.6,75443.1,80062.22,50752.54,95584.54,78872.10
9,ภาคกลาง,ชลบุรี,22286.23,21869.0,22260.0,24052.199929,23007.2,28366.5,27256.7,27665.39,...,128114.0,139319.0,191149.0,186509.0,159083.7,149191.5,170022.64,124323.25,244684.86,282402.49


In [4]:
df_panel.head(30)

,Province,Year,Income,Debt,Annual_Income,DTI,Label,Year_prev,gap,Debt_lag1,Income_lag1,DTI_lag1,Debt_growth_ann,Income_growth_ann,Region_กรุงเทพมหานครและปริมณฑล,Region_ภาคกลาง,Region_ภาคตะวันออกเฉียงเหนือ,Region_ภาคเหนือ,Region_ภาคใต้
0,กระบี่,2547,16876.960000,201625.17,202523.520000,0.995564,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True
1,กระบี่,2549,18446.000000,156798.00,221352.000000,0.708365,0,2547.0,2.0,201625.17,16876.960000,0.995564,NaN,NaN,False,False,False,False,True
2,กระบี่,2550,18852.000000,182167.00,226224.000000,0.805251,0,2549.0,1.0,156798.00,18446.000000,0.708365,-0.118144,0.045452,False,False,False,False,True
3,กระบี่,2552,23647.368327,219674.00,283768.419922,0.774131,0,2550.0,2.0,182167.00,18852.000000,0.805251,0.161794,0.022010,False,False,False,False,True
4,กระบี่,2554,33350.300000,177494.60,400203.600000,0.443511,0,2552.0,2.0,219674.00,23647.368327,0.774131,0.098132,0.119986,False,False,False,False,True
5,กระบี่,2556,27275.800000,168945.80,327309.600000,0.516165,0,2554.0,2.0,177494.60,33350.300000,0.443511,-0.101117,0.187568,False,False,False,False,True
6,กระบี่,2558,31011.500000,373325.20,372138.000000,1.003190,1,2556.0,2.0,168945.80,27275.800000,0.516165,-0.024379,-0.095645,False,False,False,False,True
7,กระบี่,2560,34052.870000,289237.36,408634.440000,0.707814,0,2558.0,2.0,373325.20,31011.500000,1.003190,0.486517,0.066283,False,False,False,False,True
8,กระบี่,2562,28521.690000,216585.68,342260.280000,0.632810,0,2560.0,2.0,289237.36,34052.870000,0.707814,-0.119796,0.047889,False,False,False,False,True
9,กระบี่,2564,29840.640000,318191.43,358087.680000,0.888585,0,2562.0,2.0,216585.68,28521.690000,0.632810,-0.134658,-0.084811,False,False,False,False,True


In [5]:
%pip install statsmodels
%pip install scikit-learn
%pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
# ============================================================
# DAY 2 : Baseline Modeling
# ใช้ต่อจาก Day 1: X_train, y_train, X_test, y_test, train_df, feature_cols
# ============================================================

import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import pandas as pd

# ------------------------------------------------------------
# MODEL 1A : Logistic Regression ด้วย statsmodels
# -> เป้าหมาย: ดู coefficient + p-value (นัยสำคัญทางสถิติ)
# ------------------------------------------------------------
# add_constant = เพิ่มคอลัมน์ intercept (beta_0) ให้สูตร
# ln(p/(1-p)) = b0 + b1*x1 + ... ครบตามทฤษฎี
X_train_sm = sm.add_constant(X_train)

# หมายเหตุ: บาง column เป็น bool (จาก get_dummies ตอน Day 1)
# ต้องแปลงเป็น float ก่อน ไม่งั้น statsmodels error

X_train_sm = X_train_sm.astype(float)
logit_model = sm.Logit(y_train, X_train_sm)

# cov_type='cluster' -> แก้ปัญหา standard error แคบเกินจริง
# เพราะข้อมูลจังหวัดเดียวกันคนละปีไม่ได้เป็นอิสระจากกัน (ตามที่คุยกัน)
logit_result = logit_model.fit(
    cov_type='cluster',
    cov_kwds={'groups': train_df['Province']}
)

print("=" * 60)
print("Logistic Regression (statsmodels) -- ใช้ดู coefficient/p-value")
print("=" * 60)
print(logit_result.summary())

# ตัวแปรไหน p-value < 0.05 ถือว่ามีนัยสำคัญทางสถิติ
# ดูคอลัมน์ P>|z| ใน summary ด้านบน


# ------------------------------------------------------------
# MODEL 1B : Logistic Regression ด้วย sklearn
# -> เป้าหมาย: ได้โมเดลที่ใช้ .predict_proba() จริงสำหรับ Streamlit
# ------------------------------------------------------------
# class_weight='balanced' -> แก้ปัญหา imbalanced data
# (กลุ่ม High-Risk มีสัดส่วนน้อยกว่า Stable มาก ตามที่เห็นใน Day 1)
# วิธีทำงาน: ให้น้ำหนัก error ของกลุ่มเล็กมากกว่ากลุ่มใหญ่ตามสัดส่วนผกผัน
log_reg_sklearn = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
log_reg_sklearn.fit(X_train, y_train)

print("\nsklearn Logistic Regression -- เทรนเสร็จแล้ว (ใช้สำหรับ deploy)")


# ------------------------------------------------------------
# MODEL 2 : Random Forest
# -> เป้าหมาย: จัดการ non-linear relationship ที่ Logistic จับไม่ได้
# ------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=300,      # จำนวนต้นไม้ ยิ่งเยอะยิ่งเสถียร แต่ช้าขึ้น
    max_depth=5,           # จำกัดความลึก กัน overfit (ข้อมูลมีแค่ ~700 แถว)
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

print("Random Forest -- เทรนเสร็จแล้ว")


# ------------------------------------------------------------
# MODEL 3 : LightGBM
# -> เป้าหมาย: ลอง gradient boosting เทียบกับ RF
# ------------------------------------------------------------
# is_unbalance=True คือ class_weight='balanced' เวอร์ชันของ LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    is_unbalance=True,
    random_state=42,
    verbose=-1             # ปิด log รก ๆ ตอนเทรน
)
lgb_model.fit(X_train, y_train)

print("LightGBM -- เทรนเสร็จแล้ว")


# ------------------------------------------------------------
# STEP สุดท้าย : เก็บโมเดลทั้งหมดไว้ใน dict เดียว
# -> สะดวกตอนเอาไปวน loop ประเมินผลใน Day 3
# ------------------------------------------------------------
trained_models = {
    'Logistic Regression': log_reg_sklearn,
    'Random Forest': rf_model,
    'LightGBM': lgb_model,
}

print("\nสรุป: เทรนครบ 3 โมเดล พร้อมไป Evaluation แล้ว")
print(list(trained_models.keys()))

         Current function value: 0.119529
         Iterations: 35
Logistic Regression (statsmodels) -- ใช้ดู coefficient/p-value
                           Logit Regression Results                           
Dep. Variable:                  Label   No. Observations:                  611
Model:                          Logit   Df Residuals:                      601
Method:                           MLE   Df Model:                            9
Date:                Thu, 02 Jul 2026   Pseudo R-squ.:                  0.2290
Time:                        23:35:28   Log-Likelihood:                -73.032
converged:                      False   LL-Null:                       -94.728
Covariance Type:              cluster   LLR p-value:                 1.826e-06
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                            -11.0265

/Users/hary/Library/Python/3.9/lib/python/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/hary/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hary/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hary/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hary/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_poi

LightGBM -- เทรนเสร็จแล้ว

สรุป: เทรนครบ 3 โมเดล พร้อมไป Evaluation แล้ว
['Logistic Regression', 'Random Forest', 'LightGBM']
